# The Coefficient of Determination: What Does $R^2$ Actually Mean?

So far in this course we have:
- Learned how to measure the **linear association** between two variables using Pearson's correlation coefficient $r$
- Seen how **linear regression** emerges cleanly from $r$ — the slope and intercept of the regression line are both derived from it

But a natural question remains: **how good is our regression line, really?**

A regression line can always be computed, even when the relationship between two variables is weak or noisy. We need a way to evaluate how much our line actually improves our predictions. That's where $R^2$ — the **coefficient of determination** — comes in.

**By the end of this notebook you will be able to:**
1. Explain intuitively what $R^2$ measures
2. Calculate $SS_{\text{Total}}$ — the sum of squared errors around the mean
3. Calculate $SS_{\text{Residual}}$ — the sum of squared errors around the regression line
4. Compute $R^2$ and interpret its value
5. Apply these steps to a real dataset

In [ ]:
# Run this cell first — it loads the tools we need
import numpy as np
import matplotlib.pyplot as plt
from datascience import *
%matplotlib inline

## Part 1: Our Helper Functions

These are the same functions we've been using throughout the course. Run the cell to load them.

In [ ]:
def standard_units(any_numbers):
    "Convert any array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

## Part 2: The Core Intuition — Two Baselines

### The Dumb Baseline: Just Guess the Mean

Imagine someone hands you a list of Y values in a **random order** and asks you to predict each one before it's revealed. You have **no other information** — no X values, nothing.

What's your best strategy?

> **Guess the mean every time.**

This minimizes your total squared error. The mean is the single number that sits closest to all the data points simultaneously (in a squared-error sense). A horizontal line through the mean is literally the best you can do with zero information.

### The Smart Baseline: Use the Regression Line

Now suppose you're also given the X value before making each guess. You can use the regression line $\hat{y} = mx + b$ to make a more informed prediction.

The question $R^2$ answers is:

> **By how much did our regression line reduce the error compared to just guessing the mean?**

$$R^2 = 1 - \frac{\text{Error that remains after using regression}}{\text{Error we started with (using only the mean)}}$$

$$R^2 = 1 - \frac{SS_{\text{Residual}}}{SS_{\text{Total}}}$$

| $R^2$ value | Meaning |
|---|---|
| $1.0$ | Perfect fit — regression passes through every point |
| Between 0 and 1 | Regression explains that fraction of the total variance |
| $0.0$ | Regression is no better than guessing the mean |
| Negative | Regression is *worse* than guessing the mean (possible with a forced intercept) |

## Part 3: A Worked Example

We'll use the following dataset: $x = 0, 1, 2, 3, 4, 5, 6$ and $y = x^2$.

The true relationship is quadratic, so a *linear* regression line won't be a perfect fit — which makes it a useful case for exploring $R^2$.

### Step 3.1 — Build the Table

In [ ]:
x_vals = np.arange(0, 7)     # 0, 1, 2, 3, 4, 5, 6
y_vals = x_vals ** 2         # 0, 1, 4, 9, 16, 25, 36

data = Table().with_columns(
    'x', x_vals,
    'y', y_vals
)
data

### Step 3.2 — Visualize the Data and the Mean

The red dashed line is our "dumb baseline" — if we had to predict every y value without any x information, this is the best we could do.

In [ ]:
y_mean = np.mean(data.column('y'))
print(f"Mean of y: {y_mean}")

plt.figure(figsize=(8, 5))
plt.scatter(data.column('x'), data.column('y'), color='steelblue', s=80, zorder=5, label='Data points')
plt.axhline(y=y_mean, color='tomato', linewidth=2, linestyle='--', label=f'Mean of y = {y_mean}')

# Vertical lines show the error at each point if we predict the mean
for xi, yi in zip(data.column('x'), data.column('y')):
    plt.plot([xi, xi], [yi, y_mean], color='tomato', alpha=0.4, linewidth=1.5)

plt.xlabel('x')
plt.ylabel('y')
plt.title('Data and the Mean Baseline — Error Before Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Each red vertical segment is the error we'd make at that point if we just predicted the mean. $SS_{\text{Total}}$ is the sum of all those segments **squared**.

### Step 3.3 — Calculate $SS_{\text{Total}}$

$$SS_{\text{Total}} = \sum_{i=1}^{n} (y_i - \bar{y})^2$$

In [ ]:
deviations_from_mean = data.column('y') - y_mean
squared_deviations   = deviations_from_mean ** 2

ss_total_table = Table().with_columns(
    'x',            data.column('x'),
    'y',            data.column('y'),
    'y - mean',     deviations_from_mean,
    '(y - mean)^2', squared_deviations
)
ss_total_table

In [ ]:
SS_total = np.sum(squared_deviations)
print(f"SS_Total = {SS_total}")

You should get **SS_Total = 1092**, matching the reading.

---

### Step 3.4 — Fit the Regression Line

In [ ]:
m = slope(data, 'x', 'y')
b = intercept(data, 'x', 'y')

print(f"Slope:     {m:.4f}")
print(f"Intercept: {b:.4f}")
print(f"Regression line: y = {m:.2f}x + ({b:.2f})")

You should get approximately **y = 6x − 5**, matching the reading. Now let's plot that regression line as a prediction for y.

In [ ]:
y_predicted = m * data.column('x') + b

plt.figure(figsize=(8, 5))
plt.scatter(data.column('x'), data.column('y'), color='steelblue', s=80, zorder=5, label='Data points')
plt.plot(data.column('x'), y_predicted, color='seagreen', linewidth=2, label=f'Regression: y = {m:.1f}x + ({b:.1f})')
plt.axhline(y=y_mean, color='tomato', linewidth=2, linestyle='--', label=f'Mean = {y_mean}', alpha=0.5)

# Green segments = residuals (error remaining after regression)
for xi, yi, yhat in zip(data.column('x'), data.column('y'), y_predicted):
    plt.plot([xi, xi], [yi, yhat], color='seagreen', alpha=0.5, linewidth=1.5)

plt.xlabel('x')
plt.ylabel('y')
plt.title('Regression Line and Residuals — Error After Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Each green segment is a residual: the error that REMAINS after using the regression line.")

### Step 3.5 — Calculate $SS_{\text{Residual}}$

$$SS_{\text{Residual}} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
residuals         = data.column('y') - y_predicted # take each y value and subptract the predicted value
squared_residuals = residuals ** 2 # square the residuals (so + and - don't cancel out)

ss_residual_table = Table().with_columns(
    'x',             data.column('x'),
    'y',             data.column('y'),
    'y_hat',         y_predicted,
    'y - y_hat',     residuals,
    '(y-y_hat)^2',   squared_residuals
)
ss_residual_table

In [ ]:
SS_residual = np.sum(squared_residuals)
print(f"SS_Residual = {SS_residual:.2f}")

You should get **SS_Residual ≈ 84**, matching the reading.

---

### Step 3.6 — Compute $R^2$

$$R^2 = 1 - \frac{SS_{\text{Residual}}}{SS_{\text{Total}}}$$

In [ ]:
R_squared = 1 - (SS_residual / SS_total)

print(f"SS_Total    = {SS_total:.2f}")
print(f"SS_Residual = {SS_residual:.2f}")
print(f"Fraction of error remaining: {SS_residual/SS_total:.4f}")
print(f"R^2 = {R_squared:.4f}")

You should get **R² ≈ 0.9231**, matching the reading.

**Interpretation:** Our linear regression line explains about **92.3%** of the variance in y. Using the x-values reduces our squared prediction error by 92.3% compared to just guessing the mean every time.

The remaining 7.7% of error persists because the true relationship is *quadratic* — a straight line can't perfectly capture the curve.

---
## Part 4: The Connection to Pearson's $r$

Here's a beautiful fact: for simple linear regression (one x variable),

$$R^2 = r^2$$

where $r$ is the Pearson correlation coefficient we've been computing all along. Let's verify this.

In [ ]:
r = correlation(data, 'x', 'y')

print(f"Pearson r       = {r:.4f}")
print(f"r squared       = {r**2:.4f}")
print(f"R^2 (from SS)   = {R_squared:.4f}")
print()
print("Are they equal?", np.isclose(r**2, R_squared))

This is not a coincidence. When you square Pearson's $r$, you get exactly the proportion of variance in $y$ that is explained by a linear function of $x$. This is why $R^2$ is sometimes called "the square of the correlation coefficient" — for simple linear regression, they are literally the same quantity.

---
## Part 5: Apply It to Real Data — Back to Cricket Thermometers!

As we've seen in earlier classes, as temperature rises, crickets chirp faster. Let's see how well a linear regression captures this relationship, and use $R^2$ to evaluate the fit.

### Step 5.1 — Load the Data

In [ ]:
cricket = Table.read_table("./data/cricket_thermometer.csv")
cricket

The two columns we'll work with are:
- `Chirps_per_sec` — the x variable (predictor)
- `Temperature_deg_F` — the y variable (response)

### Step 5.2 — Plot the Data and the Mean

The mean temperature has been computed for you. Replace the two `...` to add it to the plot.

In [ ]:
temp_mean = np.mean(cricket.column('Temperature_deg_F'))
print(f"Mean temperature: {temp_mean:.2f} F")

plt.figure(figsize=(8, 5))
plt.scatter(cricket.column('Chirps_per_sec'), cricket.column('Temperature_deg_F'),
            color='steelblue', s=60, zorder=5, label='Observations')

# here, plot the mean temperature as a horizontal line using axhline
plt.axhline(y=...,     
            color='tomato', linewidth=2, linestyle='--',
            label=f'Mean temp = {temp_mean:.1f} F')

plt.xlabel('Chirps per Second')
plt.ylabel('Temperature (F)')
plt.title('Cricket Chirp Rate vs. Temperature')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Step 5.3 — Calculate $SS_{\text{Total}}$

The structure is the same as Part 3. Fill in the two `...` to compute deviations and squared deviations.

In [ ]:
temp_deviations = cricket.column('Temperature_deg_F') - ...   # subtract the mean temperature
temp_sq_devs    = ... ** 2  # square the deviations from the mean

ss_total_cricket = Table().with_columns(
    'Chirps_per_sec',    cricket.column('Chirps_per_sec'),
    'Temperature_deg_F', cricket.column('Temperature_deg_F'),
    'y - mean',          temp_deviations,
    '(y - mean)^2',      temp_sq_devs
)
ss_total_cricket

In [ ]:
SS_total_cricket = np.sum(temp_sq_devs)
print(f"SS_Total = {SS_total_cricket:.4f}")

### Step 5.4 — Fit the Regression Line

Use `slope()` and `intercept()`. Both functions need the table and both column label strings. Fill in the missing y-label `...` in each call.

In [ ]:
m_cricket = slope(cricket, 'Chirps_per_sec', ...)       # fill in y label string
b_cricket = intercept(cricket, 'Chirps_per_sec', ...)   # fill in y label string

print(f"Slope:     {m_cricket:.4f}")
print(f"Intercept: {b_cricket:.4f}")
print(f"Regression line: Temperature = {m_cricket:.2f} * Chirps_per_sec + {b_cricket:.2f}")

In [ ]:
# use slope and intercept calculated above to predict the temperature 
temp_predicted = ... * cricket.column('Chirps_per_sec') + ... 

plt.figure(figsize=(8, 5))
plt.scatter(cricket.column('Chirps_per_sec'), cricket.column('Temperature_deg_F'),
            color='steelblue', s=60, zorder=5, label='Observations')
plt.plot(cricket.column('Chirps_per_sec'), temp_predicted,
         color='seagreen', linewidth=2, label='Regression line')
plt.axhline(y=temp_mean, color='tomato', linewidth=2, linestyle='--',
            label=f'Mean = {temp_mean:.1f} F', alpha=0.5)

for xi, yi, yhat in zip(cricket.column('Chirps_per_sec'),
                         cricket.column('Temperature_deg_F'), temp_predicted):
    plt.plot([xi, xi], [yi, yhat], color='seagreen', alpha=0.4, linewidth=1)

plt.xlabel('Chirps per Second')
plt.ylabel('Temperature (F)')
plt.title('Cricket Data: Regression Line and Residuals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Step 5.5 — Calculate $SS_{\text{Residual}}$

Same pattern as Part 3. Fill in the two `...` to compute residuals and their squares.

In [ ]:
temp_residuals = cricket.column('Temperature_deg_F') - ...   # subtract the predicted temperature (not the mean!)
temp_sq_resids = ... ** 2                                     # square the residuals

ss_resid_cricket = Table().with_columns(
    'Chirps_per_sec',    cricket.column('Chirps_per_sec'),
    'Temperature_deg_F', cricket.column('Temperature_deg_F'),
    'y_hat',             temp_predicted,
    'y - y_hat',         temp_residuals,
    '(y - y_hat)^2',     temp_sq_resids
)
ss_resid_cricket

In [ ]:
SS_residual_cricket = np.sum(temp_sq_resids)
print(f"SS_Residual = {SS_residual_cricket:.4f}")

### Step 5.6 — Compute $R^2$ and Verify with Pearson's $r$

Fill in the $R^2$ formula using the two SS variables you just computed.

In [ ]:
R2_cricket = 1 - (... / ...)   # SS residuals from regression / SS residuals total
r_cricket  = correlation(cricket, 'Chirps_per_sec', 'Temperature_deg_F')

print(f"SS_Total              = {SS_total_cricket:.4f}")
print(f"SS_Residual           = {SS_residual_cricket:.4f}")
print(f"R^2 (from SS)         = {R2_cricket:.4f}")
print(f"r^2 (from Pearson r)  = {r_cricket**2:.4f}")
print()
print("Do they match?", np.isclose(R2_cricket, r_cricket**2))

---
## Discussion Questions

Answer the following questions on your warmup sheet.

---

**Question 1.** In your own words, what does $SS_{\text{Total}}$ represent? Why is the mean — rather than any other value — used as the baseline in its formula?


**Question 2.** Interpret the $R^2$ value you computed for the cricket dataset. What does it tell you about how useful chirp rate is for predicting temperature? What does the remaining $(1 - R^2)$ represent?

**Question 3.** In the worked example (Part 3), we fit a *linear* line to *quadratic* data and still got $R^2 \approx 0.92$. Does a high $R^2$ guarantee that your model is correct? What else should you look at when evaluating a regression model?

**Question 4 (Challenge).** The reading explains that $R^2$ can sometimes be *negative*. The cell below uses the cricket data but forces the intercept to 0. Run it, then explain in the Markdown cell why the result is negative.

In [ ]:
# Challenge — a deliberately bad regression
m_bad = m_cricket    # keep the fitted slope ...
b_bad = 0            # ... but force the intercept to 0

temp_predicted_bad = m_bad * cricket.column('Chirps_per_sec') + b_bad
residuals_bad      = cricket.column('Temperature_deg_F') - temp_predicted_bad
SS_residual_bad    = np.sum(residuals_bad ** 2)

R2_bad = 1 - (SS_residual_bad / SS_total_cricket)

print(f"SS_Total                      = {SS_total_cricket:.2f}")
print(f"SS_Residual (forced b=0)      = {SS_residual_bad:.2f}")
print(f"R^2 with forced intercept = 0 : {R2_bad:.4f}")

*Why is R² negative when we force the intercept to 0?*